## Setup

In [2]:
import os
import numpy as np
import pandas as pd

RESPONSE_WINDOW_MIN = 10
LOOKBACK_MIN = 30
RECENT_N = 5
STUDY_APP = "org.pielot.sns2"

user_files = sorted(f for f in os.listdir("mobile_phone_use/data") if f.endswith(".csv"))
print("User files:", len(user_files))

User files: 342


## Profile Table

In [25]:
PROFILE_COLS = ["uuid", "age", "gender"]

profile = pd.read_csv("mobile_phone_use/pinfo.csv", usecols=PROFILE_COLS)
profile["uuid"] = profile["uuid"].str.strip().str.lower()
# profile = profile.drop_duplicates(subset="uuid").reset_index(drop=True) ########## Removed
profile.columns = [c.lower() for c in profile.columns]
print(profile.shape)

(342, 3)


## Raw Sensor Files: Loading and Text Standardization

In [4]:
sensor_columns = ["uuid", "ts", "sensor_id", "WorkingDay",
               "Esm_Posted", "Esm_Timedout", "Esm_Clicked", "Esm_Responded",
               "Screen_Value", "Loc_Type", "UserAct_Type", "App_Value",
               "Notif_Action", "Notif_App", "Batt_Value", "Charging_Value",
               "Light_Value", "Noise_Value", "Acc_Avg", "Wifi_Conn"]

textual_columns = ["sensor_id", "Screen_Value", "Loc_Type", "UserAct_Type",
             "App_Value", "Notif_Action", "Notif_App"]


def clean_text(series):
    s = series.astype("string").str.strip().str.lower()
    s = s.str.replace(r"\s*&\s*", " and ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.replace(r"^[+-](?=[a-z])", "", regex=True)
    return s


def load_user(path):
    df = pd.read_csv(path, usecols=sensor_columns, low_memory=False)
    for col in textual_columns:
        df[col] = clean_text(df[col])
    df["uuid"] = df["uuid"].str.strip().str.lower()
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df = df.dropna(subset=["ts", "sensor_id"]).sort_values("ts", kind="stable").reset_index(drop=True)
    return df


def is_value(series, value):
    return series.fillna("").eq(value).to_numpy(dtype=bool)


def has_value(series):
    return series.notna().to_numpy(dtype=bool)

# load_user("mobile_phone_use/data/u000.csv")

## Step 1: Extract Each ESM Notification Event

In [10]:
# def utc_offset(df):
#     other = df.loc[df["sensor_id"] != "esm"]
#     gap_min = (other["ts"] - pd.to_datetime(other["ts_raw"], unit="ms")).dt.total_seconds() / 60
#     gap_min = (gap_min / 30).round() * 30
#     return pd.Timedelta(minutes=float(gap_min.mode().iloc[0]))


def extract_events(df):
    # offset = utc_offset(df)
    esm = df.loc[df["sensor_id"] == "esm"].copy()
    
    for col in ["Esm_Posted", "Esm_Timedout", "Esm_Clicked", "Esm_Responded"]:
        esm[col] = pd.to_datetime(esm[col],  format='mixed', errors="coerce")
        
    esm = esm.dropna(subset=["Esm_Posted"]).drop_duplicates(subset=["Esm_Posted"])
    events = esm.rename(columns={"Esm_Posted": "notif_time", "Esm_Clicked": "click_time",
                             "Esm_Responded": "response_time", "Esm_Timedout": "timeout_time",
                             "WorkingDay": "working_day"})
    
    events = events[["uuid", "notif_time", "click_time", "response_time", "timeout_time",
             "working_day"]].reset_index(drop=True)

    def minutes(later): return (later - events["notif_time"]).dt.total_seconds() / 60
        
    events["click_delay_min"] = minutes(events["click_time"])
    events["response_delay_min"] = minutes(events["response_time"])
    events["timeout_delay_min"] = minutes(events["timeout_time"])
    
    events["clicked"] = events["click_time"].notna()
    events["timed_out"] = events["timeout_time"].notna()

    # end_time = ev["response_time"].fillna(ev["timeout_time"])
    # raw_local = pd.to_datetime(ev["ts_raw"], unit="ms") + offset
    # ev["clock_gap_sec"] = (end_time - raw_local).dt.total_seconds().abs()

    # gap_ok = ev["clock_gap_sec"].isna() | (ev["clock_gap_sec"] <= CLOCK_TOLERANCE_SEC)
    delays_ok = np.logical_and.reduce([
        events[c].isna() | (events[c] >= 0) for c in ["click_delay_min", "response_delay_min", "timeout_delay_min"]
    ])
    events["event_time_ok"] = delays_ok

    events["responded"] = events["response_delay_min"].between(0, RESPONSE_WINDOW_MIN).astype(int)
    return events

# extract_events(load_user("mobile_phone_use/data/u000.csv"))

In [8]:
STATE_LOOKBACK_MIN = 120

LAST_VALUE_COLS = {
    "Screen_Value": ("screen_state", STATE_LOOKBACK_MIN),
    "Loc_Type": ("location_type", LOOKBACK_MIN),
    "UserAct_Type": ("activity_type", STATE_LOOKBACK_MIN),
    "App_Value": ("last_app", STATE_LOOKBACK_MIN),
    "Batt_Value": ("battery_level", LOOKBACK_MIN),
    "Charging_Value": ("is_charging", LOOKBACK_MIN),
    "Light_Value": ("light_level", STATE_LOOKBACK_MIN),
    "Noise_Value": ("noise_level", LOOKBACK_MIN),
    "Acc_Avg": ("acc_avg", LOOKBACK_MIN),
    "Wifi_Conn": ("wifi_connected", LOOKBACK_MIN),
}


def add_sensor_features(events, df):
    events = events.sort_values("notif_time").reset_index(drop=True)

    for source, (name, minutes) in LAST_VALUE_COLS.items():
        sub = df.loc[df[source].notna(), ["ts", source]]
        if source == "App_Value":
            sub = sub.loc[sub[source] != STUDY_APP]
        if sub.empty:
            events[name] = np.nan
            continue
        sub = sub.rename(columns={source: name})
        events = pd.merge_asof(events, sub, left_on="notif_time", right_on="ts",
                           direction="backward", tolerance=pd.Timedelta(minutes=minutes),
                           allow_exact_matches=False).drop(columns="ts")

    event_times = events["notif_time"].to_numpy()
    window = np.timedelta64(LOOKBACK_MIN, "m")

    def count_before(mask):
        times = df.loc[mask, "ts"].to_numpy()
        high = np.searchsorted(times, event_times, side="left")
        low = np.searchsorted(times, event_times - window, side="left")
        return high - low

    not_study_notif = ~is_value(df["Notif_App"], STUDY_APP)
    not_study_app = ~is_value(df["App_Value"], STUDY_APP)

    events[f"notifications_{LOOKBACK_MIN}m"] = count_before(
        is_value(df["Notif_Action"], "posted") & has_value(df["Notif_App"]) & not_study_notif)
    events[f"app_events_{LOOKBACK_MIN}m"] = count_before(has_value(df["App_Value"]) & not_study_app)
    events[f"unlocks_{LOOKBACK_MIN}m"] = count_before(is_value(df["Screen_Value"], "unlocked"))
    return events


def add_time_features(events):
    events["hour"] = events["notif_time"].dt.hour
    events["day_of_week"] = events["notif_time"].dt.dayofweek
    time_period_conditions = [
        events['hour'].between(5, 6),
        events['hour'].between(7, 9),
        events['hour'].between(10, 11),
        events['hour'].between(12, 15),
        events['hour'].between(16, 18),
        (events['hour'] >= 19) | (events['hour'] <= 4)
    ]

    periods = ['early morning', 'mid morning', 'late morning', 'afternoon', 'evening', 'night']
    events["time_slot"] = np.select(time_period_conditions, periods, default="evening")
    return events


def process_user(path):
    df = load_user(path)
    events = extract_events(df)
    if events.empty:
        return event
    events = add_sensor_features(events, df)
    return add_time_features(events)

In [40]:
parts = []
for i, name in enumerate(user_files, 1):
    parts.append(process_user(os.path.join("mobile_phone_use/data", name)))
    print(f"{i}/{len(user_files)} users processed | ")
    if i % 100 == 0:
        print(i, "", len(user_files))
        print(f"{i}/{len(user_files)} users processed | ")

events = pd.concat([p for p in parts if len(p)], ignore_index=True)
events = events.drop_duplicates(subset=["uuid", "notif_time"]).sort_values(["uuid", "notif_time"]).reset_index(drop=True)

print("Notification events:", events.shape[0])
print("Users:", events["uuid"].nunique())
print("Clock-consistent events:", int(events["event_time_ok"].sum()))
print("Excluded for clock inconsistency:", int((~events["event_time_ok"]).sum()))
events.to_csv("esm_events.csv", index=False)

1/342 users processed | 
2/342 users processed | 
3/342 users processed | 
4/342 users processed | 
5/342 users processed | 
6/342 users processed | 
7/342 users processed | 
8/342 users processed | 
9/342 users processed | 
10/342 users processed | 
11/342 users processed | 
12/342 users processed | 
13/342 users processed | 
14/342 users processed | 
15/342 users processed | 
16/342 users processed | 
17/342 users processed | 
18/342 users processed | 
19/342 users processed | 
20/342 users processed | 
21/342 users processed | 
22/342 users processed | 
23/342 users processed | 
24/342 users processed | 
25/342 users processed | 
26/342 users processed | 
27/342 users processed | 
28/342 users processed | 
29/342 users processed | 
30/342 users processed | 
31/342 users processed | 
32/342 users processed | 
33/342 users processed | 
34/342 users processed | 
35/342 users processed | 
36/342 users processed | 
37/342 users processed | 
38/342 users processed | 
39/342 users processe

C:\Users\RNA\AppData\Local\Temp\ipykernel_21976\2631424289.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([p for p in parts if len(p)], ignore_index=True)


Notification events: 113008
Users: 342
Clock-consistent events: 113003
Excluded for clock inconsistency: 5


In [41]:
print("Events shape:", events.shape)
print("\nColumn data types:")
print(events.dtypes)

print("\nColumns containing only missing values:")
print(events.columns[events.isna().all()].tolist())

Events shape: (113008, 29)

Column data types:
uuid                          object
notif_time            datetime64[ns]
click_time            datetime64[ns]
response_time         datetime64[ns]
timeout_time          datetime64[ns]
working_day                    int64
click_delay_min              float64
response_delay_min           float64
timeout_delay_min            float64
clicked                         bool
timed_out                       bool
event_time_ok                   bool
responded                      int64
screen_state          string[python]
location_type         string[python]
activity_type         string[python]
last_app              string[python]
battery_level                float64
is_charging                  float64
light_level                  float64
noise_level                  float64
acc_avg                      float64
wifi_connected               float64
notifications_30m              int64
app_events_30m                 int64
unlocks_30m                 

## Step 2: Build the Training Table (One Row per Notification)

In [42]:
def prior_response_statistics(notif_t, outcome_t, responded, last_n):
    order = np.argsort(outcome_t, kind="stable")
    sorted_outcome = outcome_t[order]    
    previous_notifivations_count = np.searchsorted(sorted_outcome, notif_t, side="left")
    
    cum = np.concatenate([[0.0], np.cumsum(responded[order])])
    
    recent_notifications_count = [5 if count > last_n else count for count in previous_notifivations_count]
    overall_response_rate = np.where(previous_notifivations_count > 0,
                       cum[previous_notifivations_count] /
                       np.maximum(previous_notifivations_count, 1), np.nan)
    
    recent_response_rate = np.where(previous_notifivations_count > recent_notifications_count,
                      (cum[previous_notifivations_count] - cum[recent_notifications_count]) /
                      np.maximum(previous_notifivations_count - recent_notifications_count, 1), np.nan)
    
    return previous_notifivations_count, overall_response_rate, recent_response_rate


def history_for_user(user_records):
    user_records = user_records.copy()
    window = pd.Timedelta(minutes=RESPONSE_WINDOW_MIN)
    
    outcome = user_records["response_time"].fillna(user_records["timeout_time"]).fillna(user_records["notif_time"] + window)
    notif_t = user_records["notif_time"].to_numpy()
    outcome_t = outcome.to_numpy()
    responded = user_records["responded"].to_numpy(dtype=float)

    count, overall, recent = prior_response_statistics(notif_t, outcome_t, responded, RECENT_N)
    user_records["prev_notifications"] = count
    user_records["prev_response_rate"] = overall
    user_records["recent_response_rate"] = recent

    last = np.full(len(user_records), np.nan)    
    if len(user_records) > 1:
        last[1:] = np.where(outcome_t[:-1] < notif_t[1:], responded[:-1], np.nan)
        
    user_records["last_notification_responded"] = last
    user_records["mins_since_prev_notification"] = user_records["notif_time"].diff().dt.total_seconds() / 60

    time_slot_rate = np.full(len(user_records), np.nan)    
    for _, idx in user_records.groupby("time_slot").indices.items():
        _, slot_overall, _ = prior_response_statistics(notif_t[idx], outcome_t[idx], responded[idx], RECENT_N)
        time_slot_rate[idx] = slot_overall
        
    user_records["slot_prev_response_rate"] = time_slot_rate
    return user_records


valid = events.loc[events["event_time_ok"]].sort_values(["uuid", "notif_time"]).reset_index(drop=True)
history = pd.concat([history_for_user(user_records) for _, user_records in valid.groupby("uuid", sort=False)], ignore_index=True)
print(history.shape)

(113003, 35)


## Step 3: Merge the User Profile

In [43]:
SENSOR_FEATURES = [name for name, _ in LAST_VALUE_COLS.values()] + [
    f"notifications_{LOOKBACK_MIN}m", f"app_events_{LOOKBACK_MIN}m", f"unlocks_{LOOKBACK_MIN}m"]

HISTORY_FEATURES = ["prev_notifications", "prev_response_rate", "recent_response_rate",
                    "slot_prev_response_rate", "last_notification_responded",
                    "mins_since_prev_notification"]

TIME_FEATURES = ["hour", "day_of_week", "working_day", "time_slot"]

training = history[["uuid", "notif_time"] + TIME_FEATURES + SENSOR_FEATURES + HISTORY_FEATURES + ["responded"]]
training = training.merge(profile, on="uuid", how="left", validate="many_to_one")

profile_features = [c for c in profile.columns if c != "uuid"]
training = training[["uuid", "notif_time"] + TIME_FEATURES + SENSOR_FEATURES + HISTORY_FEATURES
                    + profile_features + ["responded"]]

print("Rows without profile:", int(training["age"].isna().sum()))

Rows without profile: 0


In [44]:
# 1. Select float columns
float_cols = training.select_dtypes(include=["float32", "float64"]).columns.tolist()

# 2. Select all numerical columns (integers and floats)
numerical_cols = training.select_dtypes(include=["number"]).columns.tolist()

# 3. Select categorical columns (text and category types)
categorical_cols = training.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# 4. Exclude identifiers, timestamps, and target from model features
exclude_cols = ["uuid", "notif_time", "responded"]

float_cols = [c for c in float_cols if c not in exclude_cols]
numerical_cols = [c for c in numerical_cols if c not in exclude_cols]
categorical_cols = [c for c in categorical_cols if c not in exclude_cols]

# Fill missing values
for col in numerical_cols:
    training[col] = training[col].fillna(training[col].median())
    if type(col) in (['Int64', 'Int32']):
        training[col].astype('Int64')

for col in float_cols:
    training[col] = training[col].round(2)

# Fill missing categorical values with the most frequent value
for col in categorical_cols:
    training[col] = training[col].fillna("Unknown")

In [45]:
training.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113003 entries, 0 to 113002
Data columns (total 28 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   uuid                          113003 non-null  object        
 1   notif_time                    113003 non-null  datetime64[ns]
 2   hour                          113003 non-null  int32         
 3   day_of_week                   113003 non-null  int32         
 4   working_day                   113003 non-null  int64         
 5   time_slot                     113003 non-null  object        
 6   screen_state                  113003 non-null  string        
 7   location_type                 113003 non-null  string        
 8   activity_type                 113003 non-null  string        
 9   last_app                      113003 non-null  string        
 10  battery_level                 113003 non-null  float64       
 11  is_charging  

## Validation and Saving

In [46]:
print(training.duplicated(subset=["uuid", "notif_time"]).sum())
print(training["responded"].isin([0, 1]).all())
print((training["prev_response_rate"].dropna().between(0, 1)).all())

print("Final shape:", training.shape)
print("Users:", training["uuid"].nunique())
print(training["responded"].value_counts())
training.to_csv("training_table.csv", index=False)

0
True
True
Final shape: (113003, 28)
Users: 342
responded
0    81265
1    31738
Name: count, dtype: int64
